In [6]:
import json
import requests

LM_STUDIO_URL = "http://localhost:1234/v1/chat/completions"
HMDB_API_URL  = "http://localhost:8000"
MODEL         = "your-model-name"  # as shown in LM Studio


# ── 1. Tool definitions (OpenAI format) ────────────────────────────────────
with open("tools/tools.json", 'r') as f:
    tools = json.load(f)


# ── 2. Dispatch: tool name → actual HMDB API call ──────────────────────────

def _fix_signals(signals: list) -> list:
    """Auto-fill missing heights whenever ppm positions are provided."""
    default_heights = {
        "s":  lambda n: [1] * n,
        "d":  lambda n: [1, 1],
        "t":  lambda n: [1, 2, 1],
        "q":  lambda n: [1, 3, 3, 1],
        "dd": lambda n: [1, 1, 1, 1],
        "m":  lambda n: [1] * n,   # equal weights for multiplets
    }
    for sig in signals:
        if "ppm" in sig and ("heights" not in sig or not sig["heights"]):
            n    = len(sig["ppm"])
            mult = sig.get("mult", "m").lower().rstrip("*")
            sig["heights"] = default_heights.get(mult, lambda n: [1] * n)(n)
    return signals


def call_hmdb_tool(name: str, args: dict) -> str:
    print(args)
    try:
        if name == "search_metabolite":
            r = requests.get(f"{HMDB_API_URL}/metabolites/search", params=args)
        elif name == "get_metabolite":
            accession = args["accession"]
            r = requests.get(f"{HMDB_API_URL}/metabolites/{accession}")
        elif name == "get_nmr_signals":
            accession = args["accession"]
            r = requests.get(f"{HMDB_API_URL}/nmr/{accession}")
        elif name == "query_nmr":
            args["signals"] = _fix_signals(args.get("signals", []))   # ← guard
            r = requests.post(f"{HMDB_API_URL}/nmr/query", json=args)
        else:
            return json.dumps({"error": f"Unknown tool: {name}"})

        r.raise_for_status()
        return json.dumps(r.json(), indent=2)

    except Exception as e:
        return json.dumps({"error": str(e)})


# ── 3. Agentic loop ────────────────────────────────────────────────────────

def chat(user_message: str, system_prompt: str = None) -> str:
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": user_message})

    print(f"\n{'─'*60}\nUser: {user_message}\n{'─'*60}")

    # Keep looping until the model stops calling tools
    while True:
        response = requests.post(
            LM_STUDIO_URL,
            json={
                "model":       MODEL,
                "messages":    messages,
                "tools":       tools,
                "tool_choice": "auto",
                "temperature": 0.2,
            }
        ).json()

        choice  = response["choices"][0]
        message = choice["message"]

        # Append assistant turn (with any tool_calls) to history
        messages.append(message)

        # If no tool calls → final answer, we're done
        if choice["finish_reason"] != "tool_calls":
            return message["content"]

        # Execute every tool call the model requested
        for tc in message.get("tool_calls", []):
            fn_name = tc["function"]["name"]
            fn_args = json.loads(tc["function"]["arguments"])

            print(f"\n→ Tool call: {fn_name}({json.dumps(fn_args, indent=2)})")
            result = call_hmdb_tool(fn_name, fn_args)
            print(f"← Result preview: {result[:300]}...")

            # Feed the result back as a tool message
            messages.append({
                "role":         "tool",
                "tool_call_id": tc["id"],
                "content":      result,
            })


# ── 4. System prompt ───────────────────────────────────────────────────────

SYSTEM = """
You are a metabolomics assistant with access to a local HMDB database.
When answering questions about metabolites or NMR spectra, always call the 
appropriate tool first and base your answer on the returned data.

## Similarity score interpretation
The similarity score returned by query_nmr indicates how well a metabolite's 
recorded NMR signals match the query:

| Score        | Match quality | Interpretation                                              |
|--------------|---------------|-------------------------------------------------------------|
| >= 2.0       | ✅ Very good  | Strong candidate — signals and multiplet shapes align well  |
| >= 1.0 < 2.0 | ⚠️ Acceptable | Partial match — some signals fit but others may be missing  |
| < 1.0        | ❌ Poor       | Weak match — likely coincidental overlap, treat with caution|

Always comment on the score quality in your interpretation.

## Response format
After receiving results, respond with:
1. A markdown table of top matches:
   | Rank | Name | Accession | Similarity | Match quality | Details |
   where Match quality is ✅ / ⚠️ / ❌ based on the score thresholds above,
   and Details is: [view](http://localhost:8000/metabolites/{accession})
2. A brief interpretation (1–2 sentences per hit) noting the score quality 
   and what it suggests about the match.
"""


In [ ]:
# ── 5. Run it ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    answer = chat(
        user_message=(
            "Give me a list of metabolites that could have a doublet around 1.8 ppm "
            "and a triplet around 4.2 ppm. The doublet has peak positions at 1.800 and 1.801, "
            "and the triplet has peaks at 4.200, 4.201, and 4.203."
        ),
        system_prompt=SYSTEM,
    )
    print(f"\n{'─'*60}\nAssistant:\n{answer}")

In [8]:
# -- Look for Lactate
if __name__ == "__main__":
    answer = chat(
        user_message=(
            "Look for all metabolites that have a quadruplet with the following peak positions [4.095,4.1065,4.118,4.1295] and a doublet with peaks [1.31, 1.324]. Guess the heights for each signal"
        ),
        system_prompt=SYSTEM,
    )
    print(f"\n{'─'*60}\nAssistant:\n{answer}")


────────────────────────────────────────────────────────────
User: Look for all metabolites that have a quadruplet with the following peak positions [4.095,4.1065,4.118,4.1295] and a doublet with peaks [1.31, 1.324]. Guess the heights for each signal
────────────────────────────────────────────────────────────

→ Tool call: query_nmr({
  "signals": [
    {
      "range": [
        4.09,
        4.13
      ],
      "mult": "q",
      "ppm": [
        4.095,
        4.1065,
        4.118,
        4.1295
      ],
      "heights": [
        1,
        3,
        3,
        1
      ]
    },
    {
      "range": [
        1.305,
        1.325
      ],
      "mult": "d",
      "ppm": [
        1.31,
        1.324
      ],
      "heights": [
        1,
        1
      ]
    }
  ]
})
{'signals': [{'range': [4.09, 4.13], 'mult': 'q', 'ppm': [4.095, 4.1065, 4.118, 4.1295], 'heights': [1, 3, 3, 1]}, {'range': [1.305, 1.325], 'mult': 'd', 'ppm': [1.31, 1.324], 'heights': [1, 1]}]}
← Result preview